# B Cell scArches-Compatible Retraining Pipeline v4.1

**Purpose**: Retrain scVI/scANVI with expert annotations for scArches compatibility

**Key Features**:
- Load full-gene data + subcluster annotations
- Force important markers into HVG
- scArches-compatible training (layer=None, weight_decay=0)
- Export HVG + UMAP operator + metadata

**Author**: r2end  
**Date**: 2026-02-01  
**Memory**: <40GB RAM

## Cell 1: Configuration and Imports

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import gc
import warnings
from scipy import sparse
from datetime import datetime
import json
import pickle

warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, dpi_save=300, frameon=False)
sc.settings.n_jobs = 48

# Thread control
import os
os.environ['OMP_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'

# GPU settings
import torch
torch.set_float32_matmul_precision('high')
scvi.settings.seed = 42

print("="*80)
print("B CELL scArches RETRAINING v4.1")
print(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

In [ ]:
# ===== Paths =====
BASE_DIR = Path(f"/home/h2048/data/py/{datetime.now().strftime('%m%d')}/bcell_scarches_v4_1")

# Input files
FULLGENE_FILE = Path('/home/h2048/data/py/0111/celltypist_bcell/adata_bcell_FINAL_corrected_20260114.h5ad')
SUBCLUSTER_FILE = Path('/home/h2048/data/py/0119/bcell_analysis/results/subcluster_v2_20260119/adata_bcell_subclustered_FINAL_v2_20260119.h5ad')
CSV_FILE = Path('/home/h2048/logs/20260202/bcell_markers_comprehensive.csv')  # UPDATE THIS

# Output dirs
OUTPUT_DIR = BASE_DIR / "results"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
EXPORT_DIR = OUTPUT_DIR / "scarches_package"

for d in [OUTPUT_DIR, FIG_DIR, MODEL_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\n[OK] Directories created:")
print(f"  - Output: {OUTPUT_DIR}")

In [ ]:
# ===== Analysis Parameters =====
BATCH_KEY = 'sample'
SUBCLUSTER_KEY = 'cell_type_L3'  # From subcluster file
NEW_CELLTYPE_KEY = 'cell_type_expert'

# CSV columns
CSV_CLUSTER_COL = 'Cluster'
CSV_CELLTYPE_COL = 'Cell_Type'
CSV_QC_FLAG_COL = 'QC_Flag'
CSV_MARKERS_COL = 'Markers'

# scVI/scANVI params (scArches compatible)
N_HVG = 4000
SCVI_PARAMS = {
    'n_latent': 50,
    'n_hidden': 128,
    'n_layers': 2,
    'dropout_rate': 0.1,
    'gene_likelihood': 'nb'
}

SCVI_TRAIN = {
    'max_epochs': 400,
    'batch_size': 256,
    'early_stopping': True,
    'early_stopping_patience': 20,
    'plan_kwargs': {'lr': 1e-3}
}

SCANVI_TRAIN = {
    'max_epochs': 200,
    'batch_size': 256,
    'early_stopping': True,
    'early_stopping_patience': 15,
    'plan_kwargs': {'lr': 1e-3, 'weight_decay': 0.0}  # scArches requirement
}

UNLABELED = 'Unknown'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n[OK] Configuration loaded")
print(f"  - Batch key: {BATCH_KEY}")
print(f"  - HVG: {N_HVG}")
print(f"  - scVI latent: {SCVI_PARAMS['n_latent']}")
print(f"  - weight_decay: {SCANVI_TRAIN['plan_kwargs']['weight_decay']} (scArches)")

## Cell 2: Load Full-Gene Data and Subcluster Annotations

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

# Load full-gene data
print(f"\nReading full-gene data: {FULLGENE_FILE.name}")
adata_full = sc.read_h5ad(FULLGENE_FILE)
print(f"[OK] Loaded: {adata_full.shape[0]:,} cells x {adata_full.shape[1]:,} genes")

# Load subcluster data
print(f"\nReading subcluster data: {SUBCLUSTER_FILE.name}")
adata_sub = sc.read_h5ad(SUBCLUSTER_FILE)
print(f"[OK] Loaded: {adata_sub.shape[0]:,} cells x {adata_sub.shape[1]:,} genes")

# Check overlap
common_cells = set(adata_full.obs.index) & set(adata_sub.obs.index)
print(f"\nCell ID overlap: {len(common_cells):,} cells ({len(common_cells)/len(adata_full)*100:.1f}%)")

if len(common_cells) < len(adata_full) * 0.99:
    print(f"[!] WARNING: Significant cell mismatch")
else:
    print(f"[OK] Cell IDs match well")

In [ ]:
# Transfer subcluster annotations to full-gene data
print(f"\nTransferring subcluster annotations...")

# Transfer cell_type_L3 (subcluster labels)
if SUBCLUSTER_KEY in adata_sub.obs.columns:
    adata_full.obs[SUBCLUSTER_KEY] = adata_sub.obs.loc[adata_full.obs.index, SUBCLUSTER_KEY]
    print(f"[OK] Transferred '{SUBCLUSTER_KEY}': {adata_full.obs[SUBCLUSTER_KEY].nunique()} clusters")
else:
    print(f"[X] ERROR: '{SUBCLUSTER_KEY}' not found in subcluster file")
    print(f"    Available columns: {list(adata_sub.obs.columns)}")
    raise ValueError(f"Missing {SUBCLUSTER_KEY}")

# Display cluster distribution
print(f"\nSubcluster distribution:")
cluster_counts = adata_full.obs[SUBCLUSTER_KEY].value_counts().sort_index()
for cluster, count in cluster_counts.head(10).items():
    print(f"  {cluster:<50s}: {count:6,d} cells")
if len(cluster_counts) > 10:
    print(f"  ... and {len(cluster_counts)-10} more clusters")

gc.collect()

## Cell 3: Load CSV and Filter Clusters

In [ ]:
print("="*80)
print("LOADING CSV ANNOTATIONS")
print("="*80)

# Load CSV
print(f"\nReading CSV: {CSV_FILE.name}")
csv_df = pd.read_csv(CSV_FILE)
print(f"[OK] Loaded {len(csv_df)} annotations")

print(f"\nCSV columns: {list(csv_df.columns)}")
print(f"\nCSV preview:")
print(csv_df[[CSV_CLUSTER_COL, CSV_CELLTYPE_COL, CSV_QC_FLAG_COL]].head(10).to_string(index=False))

# Check QC flags
print(f"\nQC_Flag distribution:")
qc_counts = csv_df[CSV_QC_FLAG_COL].value_counts()
for flag, count in qc_counts.items():
    print(f"  {flag}: {count} clusters")

In [ ]:
# Identify clusters to remove
remove_flags = ['DOUBLET_REMOVE', 'REMOVE', 'EXCLUDE']
clusters_to_remove = csv_df[csv_df[CSV_QC_FLAG_COL].isin(remove_flags)][CSV_CLUSTER_COL].tolist()

if clusters_to_remove:
    print(f"\n[!] Clusters marked for removal ({len(clusters_to_remove)}):")
    for cluster in clusters_to_remove:
        flag = csv_df[csv_df[CSV_CLUSTER_COL]==cluster][CSV_QC_FLAG_COL].values[0]
        if cluster in adata_full.obs[SUBCLUSTER_KEY].values:
            count = (adata_full.obs[SUBCLUSTER_KEY] == cluster).sum()
            print(f"  - {cluster}: {count:,} cells (Flag: {flag})")
else:
    print(f"\n[OK] No clusters marked for removal")

# Filter
if clusters_to_remove:
    n_before = len(adata_full)
    keep_mask = ~adata_full.obs[SUBCLUSTER_KEY].isin(clusters_to_remove)
    adata = adata_full[keep_mask].copy()
    n_after = len(adata)
    
    print(f"\nFiltering results:")
    print(f"  - Before: {n_before:,} cells")
    print(f"  - After: {n_after:,} cells")
    print(f"  - Removed: {n_before-n_after:,} cells ({(n_before-n_after)/n_before*100:.2f}%)")
else:
    adata = adata_full.copy()
    print(f"\n[OK] No filtering applied")

del adata_full
gc.collect()

## Cell 4: Apply Expert Annotations from CSV

In [ ]:
print("="*80)
print("APPLYING EXPERT ANNOTATIONS")
print("="*80)

# Create mapping (only use OK/REVIEW clusters)
good_flags = ['OK', 'REVIEW']
csv_good = csv_df[csv_df[CSV_QC_FLAG_COL].isin(good_flags)].copy()
cluster_to_celltype = dict(zip(csv_good[CSV_CLUSTER_COL], csv_good[CSV_CELLTYPE_COL]))

print(f"\nCreated mapping for {len(cluster_to_celltype)} clusters")
print(f"\nMapping preview:")
for cluster, celltype in list(cluster_to_celltype.items())[:5]:
    print(f"  {cluster:<50s} -> {celltype}")
if len(cluster_to_celltype) > 5:
    print(f"  ... and {len(cluster_to_celltype)-5} more")

# Apply mapping
adata.obs[NEW_CELLTYPE_KEY] = (
    adata.obs[SUBCLUSTER_KEY]
    .map(cluster_to_celltype)
    .fillna(UNLABELED)
)

# Force categorical
adata.obs[NEW_CELLTYPE_KEY] = adata.obs[NEW_CELLTYPE_KEY].astype('category')
if UNLABELED not in adata.obs[NEW_CELLTYPE_KEY].cat.categories:
    adata.obs[NEW_CELLTYPE_KEY] = adata.obs[NEW_CELLTYPE_KEY].cat.add_categories([UNLABELED])

# Report coverage
n_labeled = (adata.obs[NEW_CELLTYPE_KEY] != UNLABELED).sum()
n_unlabeled = (adata.obs[NEW_CELLTYPE_KEY] == UNLABELED).sum()

print(f"\nMapping coverage:")
print(f"  - Labeled: {n_labeled:,} cells ({n_labeled/len(adata)*100:.2f}%)")
print(f"  - Unlabeled: {n_unlabeled:,} cells ({n_unlabeled/len(adata)*100:.2f}%)")

print(f"\nNew cell type distribution:")
celltype_counts = adata.obs[NEW_CELLTYPE_KEY].value_counts()
for ct, count in celltype_counts.items():
    print(f"  {ct:<60s}: {count:6,d} cells")

## Cell 5: Extract Markers from CSV

In [ ]:
print("="*80)
print("EXTRACTING MARKERS FROM CSV")
print("="*80)

# Extract all markers
all_markers_csv = []

if CSV_MARKERS_COL in csv_df.columns:
    for markers_str in csv_df[CSV_MARKERS_COL].dropna():
        # Split by semicolon
        markers = [m.strip() for m in str(markers_str).split(';')]
        # Remove HALLMARK and pathway annotations
        markers = [m for m in markers if not m.startswith('HALLMARK_')]
        markers = [m for m in markers if '(' not in m]  # Remove (pathway-inferred)
        all_markers_csv.extend(markers)
    
    # Remove duplicates
    all_markers_csv = list(set(all_markers_csv))
    
    print(f"\n[OK] Extracted {len(all_markers_csv)} unique markers from CSV")
    
    # Check availability
    markers_available = [m for m in all_markers_csv if m in adata.var_names]
    markers_missing = [m for m in all_markers_csv if m not in adata.var_names]
    
    print(f"  - Available in data: {len(markers_available)} ({len(markers_available)/len(all_markers_csv)*100:.1f}%)")
    
    if markers_missing:
        print(f"  - Missing: {len(markers_missing)}")
        if len(markers_missing) <= 20:
            print(f"    {', '.join(markers_missing)}")
        else:
            print(f"    {', '.join(markers_missing[:20])}... and {len(markers_missing)-20} more")
    
    # Store for later use
    FORCE_INCLUDE_GENES = markers_available
    
else:
    print(f"[!] WARNING: No '{CSV_MARKERS_COL}' column in CSV")
    FORCE_INCLUDE_GENES = []

print(f"\n[OK] Will force {len(FORCE_INCLUDE_GENES)} markers into HVG")

## Cell 6: scArches Data Preparation

In [ ]:
print("="*80)
print("scArches DATA PREPARATION")
print("="*80)

# Ensure counts exist
if 'counts' in adata.layers:
    print(f"\n[OK] counts layer exists")
elif hasattr(adata, 'raw') and adata.raw is not None:
    print(f"\n[OK] Using raw.X as counts")
    adata.layers['counts'] = adata.raw.X
else:
    print(f"\n[X] ERROR: No counts found")
    raise ValueError("No counts layer or raw.X available")

# Set .X to counts (no copy, direct reference)
print(f"\nSetting .X to counts (scArches requirement)...")
adata.X = adata.layers['counts']
print(f"[OK] .X = counts (shared reference)")

# Ensure sparse
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)
    print(f"[OK] Converted to sparse")

# Force categorical
print(f"\nForcing categorical dtypes...")
adata.obs[BATCH_KEY] = adata.obs[BATCH_KEY].astype(str).astype('category')
adata.obs[NEW_CELLTYPE_KEY] = adata.obs[NEW_CELLTYPE_KEY].astype(str).astype('category')
print(f"[OK] {BATCH_KEY}: {adata.obs[BATCH_KEY].nunique()} categories")
print(f"[OK] {NEW_CELLTYPE_KEY}: {adata.obs[NEW_CELLTYPE_KEY].nunique()} categories")

# Check gene names
if not adata.var_names.is_unique:
    print(f"\n[!] Making gene names unique...")
    adata.var_names_make_unique()
    print(f"[OK] Gene names unique")

# Check NaN
nan_batch = adata.obs[BATCH_KEY].isna().sum()
nan_ct = adata.obs[NEW_CELLTYPE_KEY].isna().sum()
if nan_batch > 0 or nan_ct > 0:
    print(f"\n[!] Filling NaN values...")
    if nan_batch > 0:
        adata.obs[BATCH_KEY] = adata.obs[BATCH_KEY].fillna('Unknown_Batch').astype('category')
    if nan_ct > 0:
        adata.obs[NEW_CELLTYPE_KEY] = adata.obs[NEW_CELLTYPE_KEY].fillna(UNLABELED).astype('category')
    print(f"[OK] NaN filled")

print(f"\n[OK] Data preparation complete")
print(f"  - Shape: {adata.shape}")
print(f"  - Batches: {adata.obs[BATCH_KEY].nunique()}")
print(f"  - Cell types: {adata.obs[NEW_CELLTYPE_KEY].nunique()}")

gc.collect()

## Cell 7: HVG Selection with Forced Markers

In [ ]:
print("="*80)
print("HVG SELECTION (WITH FORCED MARKERS)")
print("="*80)

# HVG selection
print(f"\nSelecting {N_HVG} HVG...")

try:
    print(f"  Attempting batch-aware HVG...")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_HVG,
        batch_key=BATCH_KEY,
        subset=False,
        flavor='seurat_v3'
    )
    hvg_method = "batch-aware"
    print(f"  [OK] Batch-aware HVG successful")
except Exception as e:
    print(f"  [!] Batch-aware failed: {e}")
    print(f"  Falling back to non-batch-aware...")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_HVG,
        subset=False,
        flavor='seurat_v3'
    )
    hvg_method = "non-batch-aware"
    print(f"  [OK] Non-batch-aware HVG successful")

n_hvg = adata.var['highly_variable'].sum()
print(f"\n[OK] Selected {n_hvg:,} HVG ({hvg_method})")

In [ ]:
# Force important markers into HVG
if FORCE_INCLUDE_GENES:
    print(f"\n[!] Forcing {len(FORCE_INCLUDE_GENES)} markers into HVG...")
    
    # Count how many are already in HVG
    markers_in_hvg = sum(adata.var.loc[m, 'highly_variable'] for m in FORCE_INCLUDE_GENES if m in adata.var_names)
    markers_to_add = [m for m in FORCE_INCLUDE_GENES if m in adata.var_names and not adata.var.loc[m, 'highly_variable']]
    
    print(f"  - Already in HVG: {markers_in_hvg}")
    print(f"  - Need to add: {len(markers_to_add)}")
    
    if markers_to_add:
        # Set these markers as highly_variable
        adata.var.loc[markers_to_add, 'highly_variable'] = True
        
        # If we exceed N_HVG, remove lowest variance non-marker genes
        current_hvg = adata.var['highly_variable'].sum()
        if current_hvg > N_HVG:
            print(f"  [!] Total HVG now {current_hvg}, trimming to {N_HVG}...")
            
            # Get variance scores
            hvg_genes = adata.var[adata.var['highly_variable']].copy()
            
            # Separate markers and non-markers
            marker_mask = hvg_genes.index.isin(FORCE_INCLUDE_GENES)
            non_marker_hvg = hvg_genes[~marker_mask].copy()
            
            # Sort non-markers by variance and keep top
            if 'variances_norm' in non_marker_hvg.columns:
                non_marker_hvg = non_marker_hvg.sort_values('variances_norm', ascending=False)
            
            n_to_keep = N_HVG - len(FORCE_INCLUDE_GENES)
            genes_to_keep = list(FORCE_INCLUDE_GENES) + list(non_marker_hvg.index[:n_to_keep])
            
            # Reset highly_variable
            adata.var['highly_variable'] = False
            adata.var.loc[genes_to_keep, 'highly_variable'] = True
        
        final_hvg = adata.var['highly_variable'].sum()
        print(f"  [OK] Final HVG count: {final_hvg:,}")
        print(f"  [OK] Markers in final HVG: {sum(adata.var.loc[m, 'highly_variable'] for m in FORCE_INCLUDE_GENES if m in adata.var_names)}")
    else:
        print(f"  [OK] All markers already in HVG")
else:
    print(f"\n[OK] No markers to force (using HVG as-is)")

In [ ]:
# Store full genes in .raw (CRITICAL - must be before subsetting)
print(f"\n[!] CRITICAL: Storing full genes in .raw (shared memory)...")
adata.raw = sc.AnnData(
    X=adata.X,  # No .copy() - shared memory
    obs=adata.obs.copy(),
    var=adata.var.copy()
)
print(f"[OK] .raw created: {adata.raw.shape[0]:,} cells x {adata.raw.shape[1]:,} genes")
print(f"     Memory efficient: 0 GB overhead (shared reference)")

# Export HVG list (scArches requirement)
hvg_genes = adata.var_names[adata.var['highly_variable']].tolist()
hvg_file = EXPORT_DIR / 'hvg_genes.txt'
with open(hvg_file, 'w') as f:
    for gene in hvg_genes:
        f.write(f"{gene}\n")
print(f"\n[OK] HVG list exported: {hvg_file.name}")

# Subset to HVG
print(f"\nSubsetting to HVG for training...")
print(f"  Before: {adata.shape[1]:,} genes")
adata = adata[:, adata.var['highly_variable']].copy()
print(f"  After: {adata.shape[1]:,} genes")
print(f"  Memory reduction: ~{(1-adata.shape[1]/adata.raw.shape[1])*100:.1f}%")

gc.collect()

## Cell 8: scVI Training (Foundation)

In [ ]:
print("="*80)
print("scVI TRAINING (FOUNDATION)")
print("="*80)

# Setup scVI
print(f"\nSetting up scVI model...")
print(f"  [!] CRITICAL: layer=None (counts in .X)")

scvi.model.SCVI.setup_anndata(
    adata,
    layer=None,  # scArches requirement
    batch_key=BATCH_KEY
)

print(f"[OK] scVI setup complete")
print(f"  - Data: {adata.shape}")
print(f"  - Batch key: {BATCH_KEY} ({adata.obs[BATCH_KEY].nunique()} batches)")

# Create model
vae = scvi.model.SCVI(adata, **SCVI_PARAMS)
print(f"[OK] Model created")

# Count parameters (robust method)
try:
    n_params = vae.module.n_params
except AttributeError:
    n_params = sum(p.numel() for p in vae.module.parameters() if p.requires_grad)
print(f"  - Parameters: {n_params:,}")

# Train
print(f"\n[!] Training scVI (may take 20-40 min with GPU)...")
print(f"Started: {pd.Timestamp.now().strftime('%H:%M:%S')}")

vae.train(**SCVI_TRAIN)

print(f"Finished: {pd.Timestamp.now().strftime('%H:%M:%S')}")
print(f"[OK] scVI training complete")

# Extract latent
adata.obsm['X_scvi'] = vae.get_latent_representation()
print(f"[OK] Latent extracted: {adata.obsm['X_scvi'].shape}")

# Save model
scvi_dir = MODEL_DIR / "scvi_foundation"
vae.save(scvi_dir, overwrite=True)
print(f"[OK] Model saved: {scvi_dir.name}/")

gc.collect()

## Cell 9: scANVI Training (Semi-supervised)

In [ ]:
print("="*80)
print("scANVI TRAINING (SEMI-SUPERVISED)")
print("="*80)

# Create scANVI from scVI
print(f"\nCreating scANVI from scVI...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    unlabeled_category=UNLABELED,
    labels_key=NEW_CELLTYPE_KEY
)
print(f"[OK] scANVI created")

# Display label distribution
print(f"\nLabel distribution:")
label_counts = adata.obs[NEW_CELLTYPE_KEY].value_counts()
for label, count in label_counts.head(10).items():
    status = "[UNLABELED]" if label == UNLABELED else "[labeled]"
    print(f"  {label:<60s}: {count:6,d} {status}")
if len(label_counts) > 10:
    print(f"  ... and {len(label_counts)-10} more")

# Train
print(f"\n[!] Training scANVI (may take 15-30 min with GPU)...")
print(f"  [!] weight_decay=0.0 (scArches requirement)")
print(f"Started: {pd.Timestamp.now().strftime('%H:%M:%S')}")

lvae.train(**SCANVI_TRAIN)

print(f"Finished: {pd.Timestamp.now().strftime('%H:%M:%S')}")
print(f"[OK] scANVI training complete")

# Get predictions
adata.obs['cell_type_scanvi_pred'] = lvae.predict()
predictions_soft = lvae.predict(soft=True)
adata.obs['scanvi_confidence'] = predictions_soft.max(axis=1).values

print(f"\n[OK] Predictions generated")
print(f"  - Mean confidence: {adata.obs['scanvi_confidence'].mean():.3f}")
print(f"  - Low confidence (<0.5): {(adata.obs['scanvi_confidence']<0.5).sum():,} cells")

# Extract latent
adata.obsm['X_scanvi'] = lvae.get_latent_representation()
print(f"[OK] Latent extracted: {adata.obsm['X_scanvi'].shape}")

# Save model
scanvi_dir = MODEL_DIR / "scanvi_semisupervised"
lvae.save(scanvi_dir, overwrite=True)
print(f"[OK] Model saved: {scanvi_dir.name}/")

gc.collect()

## Cell 10: Add UMAP Operator

In [ ]:
print("="*80)
print("CREATING UMAP OPERATOR")
print("="*80)

try:
    from umap import UMAP
    
    print(f"\nCreating UMAP operator on scANVI latent...")
    umap_op = UMAP(
        n_neighbors=30,
        min_dist=0.3,
        metric='euclidean',
        random_state=RANDOM_STATE,
        n_components=2
    )
    
    umap_embedding = umap_op.fit_transform(adata.obsm['X_scanvi'])
    adata.obsm['X_umap'] = umap_embedding
    
    print(f"[OK] UMAP operator fitted")
    print(f"  - Embedding: {umap_embedding.shape}")
    
    # Save operator
    umap_file = EXPORT_DIR / 'umap_operator.pkl'
    with open(umap_file, 'wb') as f:
        pickle.dump(umap_op, f)
    print(f"[OK] UMAP operator saved: {umap_file.name}")
    
    # Test transform
    test_idx = np.random.choice(len(adata), 100, replace=False)
    test_transform = umap_op.transform(adata.obsm['X_scanvi'][test_idx])
    print(f"[OK] Transform test successful")
    
except ImportError:
    print(f"[!] WARNING: umap-learn not installed")
    print(f"  Install with: pip install umap-learn")
    print(f"  Skipping UMAP operator creation")
    
    # Compute UMAP without operator
    sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=30)
    sc.tl.umap(adata)
    print(f"[OK] UMAP computed (no operator)")

gc.collect()

## Cell 11: Visualization

In [ ]:
print("="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# UMAP by predictions
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

sc.pl.umap(adata, color='cell_type_scanvi_pred', ax=axes[0,0], show=False,
          title='scANVI Predictions', legend_loc='right margin', frameon=False)

sc.pl.umap(adata, color='scanvi_confidence', ax=axes[0,1], show=False,
          title='Prediction Confidence', cmap='viridis', frameon=False)

sc.pl.umap(adata, color=NEW_CELLTYPE_KEY, ax=axes[1,0], show=False,
          title=f'Expert Labels ({NEW_CELLTYPE_KEY})', 
          legend_loc='right margin', frameon=False)

sc.pl.umap(adata, color=BATCH_KEY, ax=axes[1,1], show=False,
          title='Batch Integration', frameon=False)

plt.tight_layout()
umap_file = FIG_DIR / 'scanvi_umap_results.pdf'
plt.savefig(umap_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n[OK] Saved: {umap_file.name}")

gc.collect()

## Cell 12: Export Final Results

In [ ]:
print("="*80)
print("EXPORTING FINAL RESULTS")
print("="*80)

# Recover full genes
print(f"\nRecovering full gene matrix...")
adata_full = adata.raw.to_adata()
adata_full.obs = adata.obs.copy()
adata_full.obsm = adata.obsm.copy()
print(f"[OK] Full genes: {adata_full.shape}")

# Add metadata
adata_full.uns['scarches_reference'] = {
    'date': datetime.now().strftime('%Y-%m-%d'),
    'version': 'v4.1',
    'batch_key': BATCH_KEY,
    'labels_key': NEW_CELLTYPE_KEY,
    'unlabeled_category': UNLABELED,
    'n_hvg': adata.shape[1],
    'hvg_method': hvg_method,
    'n_markers_forced': len(FORCE_INCLUDE_GENES),
    'scvi_latent': SCVI_PARAMS['n_latent']
}

# Save
output_h5ad = EXPORT_DIR / f"bcell_reference_{datetime.now().strftime('%Y%m%d')}.h5ad"
print(f"\nSaving: {output_h5ad.name}")
adata_full.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)
file_size = output_h5ad.stat().st_size / 1e9
print(f"[OK] Saved: {file_size:.2f} GB")

# Export metadata
metadata = {
    'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'batch_key': BATCH_KEY,
    'labels_key': NEW_CELLTYPE_KEY,
    'unlabeled_category': UNLABELED,
    'n_hvg': adata.shape[1],
    'n_cells': adata_full.shape[0],
    'n_genes_total': adata_full.shape[1],
    'n_batches': adata_full.obs[BATCH_KEY].nunique(),
    'n_celltypes': adata_full.obs[NEW_CELLTYPE_KEY].nunique(),
    'markers_forced': len(FORCE_INCLUDE_GENES)
}

metadata_file = EXPORT_DIR / 'reference_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"[OK] Metadata: {metadata_file.name}")

# Cell type reference
celltype_ref = adata_full.obs[[NEW_CELLTYPE_KEY, 'scanvi_confidence']].copy()
celltype_ref['count'] = 1
celltype_summary = celltype_ref.groupby(NEW_CELLTYPE_KEY).agg({
    'count': 'sum',
    'scanvi_confidence': 'mean'
}).reset_index()
celltype_summary.columns = ['cell_type', 'n_cells', 'mean_confidence']
celltype_summary = celltype_summary.sort_values('n_cells', ascending=False)

celltype_file = EXPORT_DIR / 'celltype_reference.csv'
celltype_summary.to_csv(celltype_file, index=False)
print(f"[OK] Cell types: {celltype_file.name}")

print(f"\n[OK] Export complete")

## Cell 13: Summary

In [ ]:
print("="*80)
print("[OK] PIPELINE COMPLETE")
print("="*80)

print(f"""
Final Summary:
==============
- Cells: {adata_full.shape[0]:,}
- Genes (total): {adata_full.shape[1]:,}
- Genes (HVG): {adata.shape[1]:,}
- Markers forced into HVG: {len(FORCE_INCLUDE_GENES)}
- Batches: {adata_full.obs[BATCH_KEY].nunique()}
- Cell types: {adata_full.obs[NEW_CELLTYPE_KEY].nunique()}
- Mean confidence: {adata_full.obs['scanvi_confidence'].mean():.3f}

scArches Package:
=================
{EXPORT_DIR}/
  - bcell_reference_YYYYMMDD.h5ad
  - hvg_genes.txt
  - umap_operator.pkl
  - reference_metadata.json
  - celltype_reference.csv

Models:
=======
{MODEL_DIR}/
  - scvi_foundation/
  - scanvi_semisupervised/

Figures:
========
{FIG_DIR}/
  - scanvi_umap_results.pdf

Next Steps:
===========
1. Review marker expression
2. Test scArches mapping with query data
3. Perform downstream analysis
""")

print(f"Completed: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)